In [1]:
import math
import numpy as np 
import pandas as pd
from collections import defaultdict

In [2]:
class XGBoostModel():

    def __init__(self, params,random_seed=None):
        self.params = defaultdict(params)
        self.subsamples = self.params['subsamples'] \
            if self.params['subsamples'] else 0.1
        self.learning_rate = self.params['learning_rate'] \
            if self.params['learning_rate'] else 0.3
        self.base_prediction = self.params['base_score'] \
            if self.params['base_score'] else 0.5
        self.max_depth = self.params['max_depth'] \
            if self.params['max_depth'] else 5

        self.rng = np.random.default_rng(seed=random_seed)
        

In contrast to boosting in the classic GBM, instead of computing residuals between the current predictions and the target, we compute gradients and hessians of the loss function with respect to the current predictions, and instead of predicting residuals with a decision tree, we fit a special XGBoost tree booster, using the gradients and hessians

In [ ]:
def fit(self,X, y, objective, n_estimators):
    current_prediction = self.base_prediction*np.ones(shape=y.shape) #Starting mein model will produce same prediction. Then we will generate gradient and hessian and the work on it.
    self.models = []
    sample_idx = 0

    for i in range(n_estimators):
        gradients = objective.gradient(y,current_prediction) #current prediction, actual se kitna glt hain.
        hessians = objective.hessian(y, current_prediction) #rate change of the loss function

        if self.subsample == 1:
            sample_idx= None
        else: 
            sample_idx = self.rng.choice(len(y), size=math.floor(self.subsample*len(y)), replace = False)   #subsample*len(y) means total y me se kitne samples select krne hain. 
            #self.rng.choice-> randomly select the samples 

    Tree  = TreeBooster( X, gradients, hessians, self.params, self.max_depth, sample_idx) #create new trees

    current_prediction += self.learning_rate *Tree.predict(X)  #update the predictions 

    self.models.append(Tree)

def predict(self, X): 
    return (
        self.base_prediction + self.learning_rate * np.sum([tree.predict(X) for tree in self.Tree], axis = 0)
    )

XGBoostModel.fit = fit
XGBoostModel.predict = predict 

$$
\text{Base Prediction} = \text{Base Prediction} + \eta \sum_{k=1}^{K} \text{Tree}_k(X)
$$

Now we recursively build a binary tree structure by finding the best split rule for each node in the tree. The main difference is the criterion for evaluating splits and the way that we define a leaf's predicted value. Instead of being functions of the target values of the instances in each node, the criterion and predicted values are functions of the instance gradients and hessians